# The graph, and how each step built it

**Run the build first, in a terminal — not here:**

```
python build.py
```

Nothing in this notebook builds, writes, or deletes anything. Every cell reads the
finished graph and the workbench `build.py` left behind, so it runs in seconds and
can be re-run at any point without cost.

In [1]:
import json
import textwrap

from arango import ArangoClient

from sysml import config, nl

system = ArangoClient(hosts=config.ARANGO_URL).db(
    "_system", username=config.ARANGO_USER, password=config.ARANGO_PASS)
if not system.has_database(config.DB_NAME):
    raise RuntimeError(
        f"No database named {config.DB_NAME!r}. Run `python build.py` in a terminal, "
        f"then re-run this cell.")

db = config.db()


def label(doc):
    """Whatever this kind of document calls its name."""
    for key in ("entity_name", "file_name", "title"):
        if doc.get(key):
            return doc[key]
    return doc["_key"]


def show(doc, width=98):
    for key, value in doc.items():
        if key in ("_id", "_rev"):
            continue
        if key == config.EMBEDDING_FIELD:
            print(f"  {key:<16} [{len(value)} floats]")
            continue
        text = value if isinstance(value, str) else json.dumps(value, default=str)
        print(f"  {key:<16} {textwrap.shorten(text, width)}")


print(f"{config.DB_NAME} at {config.ARANGO_URL}")
for name in config.ALL_COLLECTIONS:
    print(f"  {db.collection(name).count():>7}  {name}")

dronegraph at http://localhost:8529
       30  sysml_Documents
      114  sysml_Chunks
     2225  sysml_Entities
      266  sysml_Communities
    10968  sysml_Relations


## A Document

In [2]:
show(next(iter(db.aql.execute(
    f'FOR d IN {config.DOCUMENTS} FILTER CONTAINS(d.file_name, "TechnicalComponents") '
    f'LIMIT 1 RETURN d'))))

  _key             12281052795918605159_2
  content          // // Copyright (c) 2026 AIRBUS and its affiliates. // This Source Code Form is subject to [...]
  import_number    2
  file_name        apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml
  citable_url      models/apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml
  file_ids         ["sysml:apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml"]
  models           ["apollo-11-sysml-v2"]
  files            ["apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml"]


## A Chunk

In [3]:
chunk = next(iter(db.aql.execute(
    f'FOR c IN {config.CHUNKS} FILTER CONTAINS(c.content, "S-IC") LIMIT 1 RETURN c')))
show(chunk)
print("\n  content, first 400 characters:")
print(textwrap.indent(chunk["content"][:400], "    "))

  _key             11494695660594771468_2
  tokens           1024
  chunk_order_index 1
  content          spacecraft for lunar transit and Earth return. */ part commandModule : ApolloCommandModule; [...]
  import_number    2
  embedding        [1536 floats]
  files            ["apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml"]
  models           ["apollo-11-sysml-v2"]

  content, first 400 characters:
    spacecraft for lunar transit and Earth return. */
    		part commandModule : ApolloCommandModule;
    		part serviceModule : ApolloServiceModule;
		
    		attribute :>> powerGenerated = serviceModule.powerGenerated;
    		attribute :>> powerLoad = commandModule.powerLoad + serviceModule.powerLoad;
		
    		interface cmSMUmbilical : CSMInterface connect serviceModule.umbilicalPort to commandModule.umbilicalPort;
    	}

	


## An Entity

In [4]:
show(next(iter(db.aql.execute(
    f'FOR e IN {config.ENTITIES} FILTER e.entity_name == "S-IC" RETURN e'))))

  _key             13800621249281261506_2
  entity_name      S-IC
  embedding        [1536 floats]
  import_number    2
  clusters         []
  description      A part definition that specializes RocketStage. Documented as the first stage providing [...]
  partition_id     null
  entity_type      part
  files            ["apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml"]
  models           ["apollo-11-sysml-v2"]
  source_file      apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml
  source_line      217
  attributes       {"propellantMass": {"value": 2077000, "unit": "kg"}, "dryMass": {"value": 137000, "unit": "kg"}}


## A Community

In [5]:
community = next(iter(db.aql.execute(
    f'FOR c IN {config.COMMUNITIES} FILTER c.level == 0 SORT LENGTH(c.report_string) DESC '
    f'LIMIT 1 RETURN c')))
show(community)
print("\n  report_string, first 500 characters:")
print(textwrap.indent(community["report_string"][:500], "    "))

  _key             5_2
  title            Cluster 5
  report_string    # SATURNV and Its Integrated Components This group details the SATURNV, a part definition [...]
  report_json      {"title": "SATURNV and Its Integrated Components", "summary": "This group details the [...]
  level            0
  occurrence       0.3448275862068966
  sub_communities  ["71", "69", "72", "70"]
  import_number    2
  embedding        [1536 floats]

  report_string, first 500 characters:
    # SATURNV and Its Integrated Components

    This group details the SATURNV, a part definition specializing MultistageRocket and LaunchSystem. SATURNV acts as a central assembly encompassing stages STAGE1, STAGE2, STAGE3, and INSTRUMENTUNIT, along with interfaces STAGE1TOSTAGE2 and STAGE2TOSTAGE3. These components are bound by ownership, type, and satisfy relationships linking them to multiple requirements, collectively executing the functions of a versatile launch system. The setup also includes a 


## One edge of every kind

In [6]:
for kind in db.aql.execute(
        f"FOR r IN {config.RELATIONS} COLLECT k = r.type WITH COUNT INTO n "
        f"SORT n DESC RETURN k"):
    edge = next(iter(db.aql.execute(
        f"FOR r IN {config.RELATIONS} FILTER r.type == @k LIMIT 1 RETURN r",
        bind_vars={"k": kind})))
    total = next(iter(db.aql.execute(
        f"RETURN LENGTH(FOR r IN {config.RELATIONS} FILTER r.type == @k RETURN 1)",
        bind_vars={"k": kind})))
    source, target = db.document(edge["_from"]), db.document(edge["_to"])
    extra = edge.get("relationship_type") or ""
    if edge.get("stated"):
        extra += "  stated"
    print(f"  {total:>6}  {kind:<17} {label(source)[:28]:<30} -> {label(target)[:28]:<30} {extra}")

    4737  IN_COMMUNITY      ENGINE2                        -> Cluster 8                      
    3589  RELATED_TO        BATTERY                        -> POWERMANAGEMENTMODULE          connects
    2203  MENTIONED_IN      DRONEMODELLOGICAL              -> 9720150836759968647_0          
     238  SUB_COMMUNITY_OF  Cluster 10                     -> Cluster 7                      
     114  PART_OF           9720150836759968647_0          -> DroneModelLogical.sysml        
      87  SIMILAR_TO        DRONEBATTERY_PARTS_DRONEBATT   -> DRONE_BATTERY                  


## Step 1 — `extract`

In [7]:
import networkx as nx

# out/kg is the only part of a build that is not in the database, so it can be
# missing or half-written independently of it. Say so rather than raise.
graphs = {}
for model in config.MODEL_NAMES:
    kg = config.kg(model)
    if not (kg / config.ARTIFACTS.RELATIONSHIPS).exists():
        print(f"  {model}\n    no workbench under {kg} -- rerun `python build.py`")
        continue
    graphs[model] = graph = nx.read_graphml(kg / config.ARTIFACTS.RELATIONSHIPS)
    chunks = json.loads((kg / config.ARTIFACTS.TEXT_CHUNKS).read_text(encoding="utf-8"))
    reports = json.loads((kg / config.ARTIFACTS.COMMUNITY_REPORTS).read_text(encoding="utf-8"))
    answers = json.loads((kg / "kv_store_llm_response_cache.json").read_text(encoding="utf-8"))
    n = sum(len(v) for v in answers.values()) if isinstance(answers, dict) else len(answers)
    print(f"  {model}")
    print(f"    {len(chunks):>4} chunks  {graph.number_of_nodes():>5} entities  "
          f"{graph.number_of_edges():>5} relations  {len(reports):>4} reports"
          f"   {n:>4} cached LLM answers")

sample = next((d for g in graphs.values() for n, d in g.nodes(data=True)
               if n.strip('"') == "S-IC"), None)
if sample:
    print("\n  S-IC as extraction wrote it, before the lexer reached it:")
    print(f"    fields  {sorted(sample)}")
    print("    no value, no unit, no file, no line")

used, off_ontology = {}, 0
allowed = {r.lower() for r in config.RELATIONS_ONTOLOGY}
for graph in graphs.values():
    for _, _, d in graph.edges(data=True):
        kind = (d.get("relationship_type") or "").lower()
        used[kind] = used.get(kind, 0) + 1
        off_ontology += kind not in allowed
print(f"\n  {len(used)} of {len(config.RELATIONS_ONTOLOGY)} relation types used, "
      f"{off_ontology} outside the ontology")

  DroneModelLogical
       4 chunks    112 entities    121 relations    11 reports     30 cached LLM answers
  Drone_BaseArchitecture
       1 chunks     10 entities     15 relations     2 reports      6 cached LLM answers
  apollo-11-sysml-v2
     109 chunks   1387 entities   1782 relations   253 reports    726 cached LLM answers

  S-IC as extraction wrote it, before the lexer reached it:
    fields  ['description', 'entity_type', 'source_id']
    no value, no unit, no file, no line

  14 of 18 relation types used, 0 outside the ontology


## Step 2a — `structure`

In [8]:
one = lambda aql: next(iter(db.aql.execute(aql)))  # noqa: E731

for text, aql in [
    ("entities, total", f"RETURN LENGTH(FOR e IN {config.ENTITIES} RETURN 1)"),
    ("  declared -- carries a source_file",
     f"RETURN LENGTH(FOR e IN {config.ENTITIES} FILTER e.source_file != null RETURN 1)"),
    ("  reported by the LLM only",
     f"RETURN LENGTH(FOR e IN {config.ENTITIES} FILTER e.source_file == null RETURN 1)"),
    ("  carrying an attribute value",
     f"RETURN LENGTH(FOR e IN {config.ENTITIES} "
     f"FILTER LENGTH(ATTRIBUTES(e.attributes)) > 0 RETURN 1)"),
    ("  carrying a short_name",
     f"RETURN LENGTH(FOR e IN {config.ENTITIES} FILTER e.short_name != null RETURN 1)"),
    ("RELATED_TO, total",
     f"RETURN LENGTH(FOR r IN {config.RELATIONS} FILTER r.type == 'RELATED_TO' RETURN 1)"),
    ("  stated by the syntax",
     f"RETURN LENGTH(FOR r IN {config.RELATIONS} FILTER r.stated == true RETURN 1)"),
    ("  inferred by the LLM",
     f"RETURN LENGTH(FOR r IN {config.RELATIONS} "
     f"FILTER r.type == 'RELATED_TO' AND r.stated != true RETURN 1)"),
]:
    print(f"  {one(aql):>7}  {text}")

     2225  entities, total
     1851    declared -- carries a source_file
      374    reported by the LLM only
      176    carrying an attribute value
      320    carrying a short_name
     3589  RELATED_TO, total
     3123    stated by the syntax
      466    inferred by the LLM


In [9]:
# What exact values on an exact tree buy: a rollup over declared edges only.
for row in db.aql.execute(f"""
        FOR e IN {config.ENTITIES}
          FILTER e.entity_name == "SATURNV"
          LET parts = (
            FOR child, edge IN 1..2 OUTBOUND e {config.RELATIONS}
              FILTER edge.relationship_type IN ["owns", "typedby"] AND edge.stated == true
              FILTER child.attributes.dryMass.value != null
              RETURN DISTINCT {{name: child.entity_name,
                               kg: child.attributes.dryMass.value,
                               at: CONCAT(child.source_file, ":", child.source_line)}})
          RETURN {{total: SUM(parts[*].kg), parts}}"""):
    print(f"  Saturn V dry mass   {row['total']:,} kg   from {len(row['parts'])} declared parts")
    for part in sorted(row["parts"], key=lambda p: -p["kg"]):
        print(f"    {part['kg']:>8,} kg   {part['name']:<26} {part['at']}")

  Saturn V dry mass   188,650 kg   from 4 declared parts
     137,000 kg   S-IC                       apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml:217
      36,200 kg   S-II                       apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml:201
      13,500 kg   S-IVB                      apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml:189
       1,950 kg   SATURNVINSTRUMENTUNIT      apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml:44


## Step 3 — `analogy`

In [10]:
print("  edges joining two entities with no model in common")
for row in db.aql.execute(f"""
        FOR r IN {config.RELATIONS}
          LET a = DOCUMENT(r._from), b = DOCUMENT(r._to)
          FILTER a.models != null AND b.models != null
          FILTER LENGTH(INTERSECTION(a.models, b.models)) == 0
          COLLECT kind = r.type WITH COUNT INTO n
          RETURN {{kind, n}}"""):
    print(f"    {row['n']:>5}  {row['kind']}")

print()
for row in db.aql.execute(f"""
        FOR r IN {config.RELATIONS}
          FILTER r.type == "{config.SIMILAR_TO}"
          LET a = DOCUMENT(r._from), b = DOCUMENT(r._to)
          SORT r.cosine DESC LIMIT 6
          RETURN {{cosine: r.cosine, role: r.analogy_role,
                   a: a.entity_name, am: a.models[0], b: b.entity_name, bm: b.models[0]}}"""):
    print(f"  {row['cosine']:.3f}  {row['role']:<11} "
          f"{row['a']} ({row['am']})  ~  {row['b']} ({row['bm']})")

  edges joining two entities with no model in common
       87  SIMILAR_TO

  0.823  part        DRONEBATTERY_PARTS_DRONEBATTERY (DroneModelLogical)  ~  DRONE_BATTERY (Drone_BaseArchitecture)
  0.795  part        DRONE_DRONE (DroneModelLogical)  ~  DRONE_BASEARCHITECTURE_DRONE (Drone_BaseArchitecture)
  0.786  part        DRONE_DRONE (DroneModelLogical)  ~  DRONE_SYSTEMARCHITECTURE_DRONE (Drone_BaseArchitecture)
  0.734  part        DRONEENGINE_PARTS_DRONEENGINE (DroneModelLogical)  ~  DRONE_BASEARCHITECTURE_DRONE (Drone_BaseArchitecture)
  0.728  part        DRONEENGINE_PARTS_DRONEENGINE (DroneModelLogical)  ~  DRONE_SYSTEMARCHITECTURE_DRONE (Drone_BaseArchitecture)
  0.728  part        DRONE_BASEARCHITECTURE_DRONE (Drone_BaseArchitecture)  ~  DRONEBODY_PARTS_DRONEBODY (DroneModelLogical)


## Step 4 — `examples`

In [11]:
for path, origin in ((config.AQL_EXAMPLES, "hand-written"),
                     (config.AQL_EXAMPLES_GENERATED,
                      f"written from the graph by {config.EXAMPLES_MODEL}")):
    if path.exists():
        text = path.read_text(encoding="utf-8")
        print(f"  {path.name:30} {len(text):>7,} chars   "
              f"{text.count('```aql'):>3} worked AQL examples   ({origin})")
    else:
        print(f"  {path.name:30} not present -- written by the last step of build.py")

  aql_examples.md                 26,980 chars    28 worked AQL examples   (hand-written)
  aql_examples_generated.md       26,309 chars    24 worked AQL examples   (written from the graph by gpt-5.5)


# Part 1 — AQLizer

## 1. A mass budget

In [12]:
nl.instance().ask(
    "For each Saturn V stage, give its dry mass, its propellant mass and the sum of "
    "the two, sorted by the total, with the file and line each is declared on."
).show(row_limit=7)

LLM provider initialized successfully.
Connecting to ArangoDB at http://localhost:8529 (timeout=300s)
Loaded AQL reference documentation from C:\Users\Christian\Desktop\arango\natural-language-service\txt2aql\prompts\aql_ref.md (104940 chars)
Successfully injected AQL reference into generation prompt.
Successfully injected AQL reference into fix prompt.


Q  For each Saturn V stage, give its dry mass, its propellant mass and the sum of the two, sorted by the total, with the file and line each is declared on.

AQL
   WITH sysml_Entities, sysml_Relations
   FOR e IN sysml_Entities
     FILTER e.entity_name IN ["S-IC", "S-II", "S-IVB"] // Assuming these are the Saturn V stages
     LET dryMass = e.attributes.dryMass.value
     LET propellantMass = e.attributes.propellantMass.value
     LET totalMass = dryMass + propellantMass
     FILTER dryMass != null AND propellantMass != null
     SORT totalMass DESC
     RETURN {
       stage: e.entity_name,
       dryMass: dryMass,
       propellantMass: propellantMass,
       totalMass: totalMass,
       file: e.source_file,
       line: e.source_line
     }

rows (3, first 3)
   {"stage": "S-IC", "dryMass": 137000, "propellantMass": 2077000, "totalMass": 2214000, "file": "apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml", "line": 217}
   {"stage": "S-II", "dryMass": 36200, "propellantM

## 2. What is *not* there

In [13]:
nl.instance().ask(
    "Which ten Apollo requirements have the most elements satisfying them, and how "
    "many Apollo requirements have none at all?"
).show(row_limit=3)

Loaded AQL reference documentation from C:\Users\Christian\Desktop\arango\natural-language-service\txt2aql\prompts\aql_ref.md (104940 chars)
Successfully injected AQL reference into generation prompt.
Successfully injected AQL reference into fix prompt.


Q  Which ten Apollo requirements have the most elements satisfying them, and how many Apollo requirements have none at all?

AQL
   WITH sysml_Entities, sysml_Relations
   
   LET top_satisfied_requirements = (
     FOR e IN sysml_Entities
       FILTER e.entity_type == 'requirement' AND 'apollo-11-sysml-v2' IN e.models
       FILTER e.source_file != null
       LET satisfiers = LENGTH(
         FOR r IN sysml_Relations
           FILTER r._to == e._id AND r.relationship_type == 'satisfies'
           AND r.stated == true
           RETURN 1
       )
       SORT satisfiers DESC
       LIMIT 10
       RETURN {requirement: e.entity_name, satisfiers}
   )
   
   LET unsatisfied_requirements_count = LENGTH(
     FOR e IN sysml_Entities
       FILTER e.entity_type == 'requirement' AND 'apollo-11-sysml-v2' IN e.models
       FILTER e.source_file != null
       LET satisfiers = LENGTH(
         FOR r IN sysml_Relations
           FILTER r._to == e._id AND r.relationship_type == 'satisfies'
  

## 3. The graph can be asked how it was built

In [14]:
nl.instance().ask(
    "Break the relations down by model and by whether they were read from the "
    "syntax or inferred by the LLM."
).show(row_limit=6)

Loaded AQL reference documentation from C:\Users\Christian\Desktop\arango\natural-language-service\txt2aql\prompts\aql_ref.md (104940 chars)
Successfully injected AQL reference into generation prompt.
Successfully injected AQL reference into fix prompt.


Q  Break the relations down by model and by whether they were read from the syntax or inferred by the LLM.

AQL
   WITH sysml_Entities, sysml_Relations
   FOR r IN sysml_Relations
     FILTER r.type == "RELATED_TO"
     LET from = DOCUMENT(r._from)
     FILTER from != null
     FOR model IN from.models
       COLLECT m = model, read = r.stated == true WITH COUNT INTO n
       RETURN {model: m, source: read ? "read from the syntax" : "inferred by the LLM", relations: n}

rows (6, first 6)
   {"model": "apollo-11-sysml-v2", "source": "inferred by the LLM", "relations": 447}
   {"model": "apollo-11-sysml-v2", "source": "read from the syntax", "relations": 2938}
   {"model": "Drone_BaseArchitecture", "source": "inferred by the LLM", "relations": 3}
   {"model": "Drone_BaseArchitecture", "source": "read from the syntax", "relations": 8}
   {"model": "DroneModelLogical", "source": "inferred by the LLM", "relations": 16}
   {"model": "DroneModelLogical", "source": "read from the syntax", "rel

## 4. One identifier, two elements

In [15]:
nl.instance().ask(
    "Trace HLR-R001: what satisfies it, what refines it, and what it is related "
    "to in either direction."
).show(row_limit=8)

Loaded AQL reference documentation from C:\Users\Christian\Desktop\arango\natural-language-service\txt2aql\prompts\aql_ref.md (104940 chars)
Successfully injected AQL reference into generation prompt.
Successfully injected AQL reference into fix prompt.


Q  Trace HLR-R001: what satisfies it, what refines it, and what it is related to in either direction.

AQL
   WITH sysml_Entities, sysml_Relations
   FOR e IN sysml_Entities
     FILTER e.short_name == "HLR-R001" OR e.entity_name == "HLR-R001"
     FOR v, r IN 1..1 ANY e sysml_Relations
       FILTER r.type == 'RELATED_TO'
       RETURN {of: e.entity_name, relation: r.relationship_type, other: v.entity_name,
               direction: r._from == e._id ? 'outgoing' : 'incoming',
               stated: r.stated == true, description: r.description}

rows (21, first 8)
   {"of": "HLR-R001", "relation": "satisfies", "other": "RECOVERYQUARANTINEOPERATIONS_RETRIEVECREWANDCM", "direction": "incoming", "stated": true, "description": "apollo11Mission.apollo11Phases.recoveryQuarantine.recoveryQuarantineOperations.retrieveCrewAndCM satisfies hlr-R001"}
   {"of": "HLR-R001", "relation": "owns", "other": "APOLLO11MISSIONSPECIFICATION", "direction": "incoming", "stated": true, "description": "MissionS

## 5. Joining a layer that is not in any file

In [16]:
nl.instance().ask(
    "Which requirements does the drone model state that the Apollo model has an "
    "analogous requirement for, and how close are they?"
).show(row_limit=6)

Loaded AQL reference documentation from C:\Users\Christian\Desktop\arango\natural-language-service\txt2aql\prompts\aql_ref.md (104940 chars)
Successfully injected AQL reference into generation prompt.
Successfully injected AQL reference into fix prompt.


Q  Which requirements does the drone model state that the Apollo model has an analogous requirement for, and how close are they?

AQL
   WITH sysml_Entities, sysml_Relations
   FOR e IN sysml_Entities
     FILTER 'DroneModelLogical' IN e.models
     FILTER e.entity_type == 'requirement'
     FILTER e.source_file != null
     FOR v, r IN 1..1 ANY e sysml_Relations
       FILTER r.type == 'SIMILAR_TO'
       FILTER 'apollo-11-sysml-v2' IN v.models
       FILTER v.entity_type == 'requirement'
       RETURN {
         drone_requirement: e.entity_name,
         apollo_requirement: v.entity_name,
         cosine_similarity: r.cosine,
         description: r.description
       }

rows (12, first 6)
   {"drone_requirement": "DRONEENGINESTANDARDSTAKEHOLDERREQUIREMENTS_RELIABILITY", "apollo_requirement": "MISSIONSYSTEMRELIABILITY", "cosine_similarity": 0.599417617477435, "description": "DRONEENGINESTANDARDSTAKEHOLDERREQUIREMENTS_RELIABILITY in the DroneModelLogical model plays a role like MISSIO

## The primer, measured — one question, three ways

In [17]:
PRIMERS = {
    "none":         None,                          # the deployed service, told nothing
    "generated":    config.AQL_EXAMPLES_GENERATED,  # written from the graph by build.py
    "hand-written": config.AQL_EXAMPLES,            # sysml/aql_examples.md
}


def ask(question, primer):
    """`primed=False` is the stock service: schema only, no aql_examples at all."""
    if primer is None:
        return nl.instance().ask(question, primed=False)
    return nl.instance(primer).ask(question)


def compare(question, row_limit=3):
    for name, primer in PRIMERS.items():
        if primer is not None and not primer.exists():
            print(f"\n{'=' * 78}\n{name}: {primer.name} not present -- run `python build.py`")
            continue
        print(f"\n{'=' * 78}\nPRIMER: {name}\n{'=' * 78}")
        ask(question, primer).show(row_limit=row_limit)


for name, primer in PRIMERS.items():
    size = f"{len(primer.read_text(encoding='utf-8')):,} chars" if primer and primer.exists() \
        else ("nothing" if primer is None else "not present")
    print(f"  {name:<13} {size}")

  none          nothing
  generated     26,309 chars
  hand-written  26,980 chars


In [18]:
# Provenance: the declared model, or the declared model plus whatever the LLM
# inferred? Nothing in the schema says which fields answer that.
compare("How many Apollo requirements does nothing satisfy?")

Loaded AQL reference documentation from C:\Users\Christian\Desktop\arango\natural-language-service\txt2aql\prompts\aql_ref.md (104940 chars)
Successfully injected AQL reference into generation prompt.
Successfully injected AQL reference into fix prompt.



PRIMER: none


Loaded AQL reference documentation from C:\Users\Christian\Desktop\arango\natural-language-service\txt2aql\prompts\aql_ref.md (104940 chars)
Successfully injected AQL reference into generation prompt.
Successfully injected AQL reference into fix prompt.


Q  How many Apollo requirements does nothing satisfy?

AQL
   WITH sysml_Entities, sysml_Relations, sysml_Documents, sysml_Chunks, sysml_Communities
   FOR entity IN sysml_Entities
     FILTER LENGTH(entity.files) > 0
     COLLECT WITH COUNT INTO count
   RETURN count

(  re-asked once: that query aggregates over a population without filtering it to the declared elements. Write it again, and filter the entities on `source_file != null`, so the answer covers what the files declare rather than the names the extraction step read in prose.)

rows (1, first 1)
   2225

A  The query indicates that there are 2,225 entities in the database that satisfy the condition of having associated files. This count reflects the number of entities that are declared in the files, rather than just being mentioned in prose during the extraction process.


PRIMER: generated


Loaded AQL reference documentation from C:\Users\Christian\Desktop\arango\natural-language-service\txt2aql\prompts\aql_ref.md (104940 chars)
Successfully injected AQL reference into generation prompt.
Successfully injected AQL reference into fix prompt.


Q  How many Apollo requirements does nothing satisfy?

AQL
   WITH sysml_Entities, sysml_Relations
   
   FOR req IN sysml_Entities
     FILTER req.entity_type == "requirement"
     FILTER "apollo-11-sysml-v2" IN req.models
     FILTER req.source_file != null
     LET satisfier = FIRST(
       FOR design, rel IN INBOUND req sysml_Relations
         FILTER rel.type == "RELATED_TO"
         FILTER rel.relationship_type == "satisfies"
         FILTER rel.stated == true
         LIMIT 1
         RETURN design
     )
     FILTER satisfier == null
     COLLECT WITH COUNT INTO unsatisfiedCount
     RETURN {
       model: "apollo-11-sysml-v2",
       unsatisfied_requirements: unsatisfiedCount
     }

(  re-asked once: that query aggregates over a population without filtering it to the declared elements. Write it again, and filter the entities on `source_file != null`, so the answer covers what the files declare rather than the names the extraction step read in prose, and add `AND r.stated == t

In [19]:
# Shape: the masses sit on definitions, so this only works by walking `owns` and
# `typedby` together and reading an attributes map.
compare("What is the total dry mass of the Saturn V, summed from its stages?")

Loaded AQL reference documentation from C:\Users\Christian\Desktop\arango\natural-language-service\txt2aql\prompts\aql_ref.md (104940 chars)
Successfully injected AQL reference into generation prompt.
Successfully injected AQL reference into fix prompt.



PRIMER: none


Loaded AQL reference documentation from C:\Users\Christian\Desktop\arango\natural-language-service\txt2aql\prompts\aql_ref.md (104940 chars)
Successfully injected AQL reference into generation prompt.
Successfully injected AQL reference into fix prompt.


Q  What is the total dry mass of the Saturn V, summed from its stages?

AQL
   WITH sysml_Chunks, sysml_Communities, sysml_Documents, sysml_Entities
   FOR stage IN sysml_Entities
     FILTER stage.entity_name LIKE "Saturn V Stage%"
     FOR v, e, p IN OUTBOUND stage sysml_Relations
       FILTER e.type == "PART_OF"
       COLLECT AGGREGATE totalDryMass = SUM(v.dry_mass)
   RETURN totalDryMass

rows (1, first 1)
   null

A  The query returned no rows, indicating that there is no available data on the total dry mass of the Saturn V, summed from its stages, in the current dataset.


PRIMER: generated


Loaded AQL reference documentation from C:\Users\Christian\Desktop\arango\natural-language-service\txt2aql\prompts\aql_ref.md (104940 chars)
Successfully injected AQL reference into generation prompt.
Successfully injected AQL reference into fix prompt.


Q  What is the total dry mass of the Saturn V, summed from its stages?

AQL
   WITH sysml_Entities, sysml_Relations
   
   LET q = UPPER("saturn v")
   
   LET exact = (
     FOR e IN sysml_Entities
       FILTER e.entity_name == q OR e.short_name == "saturn v"
       LIMIT 20
       RETURN e
   )
   
   LET prefixed = (
     FOR e IN sysml_Entities
       FILTER LENGTH(exact) == 0
       FILTER LIKE(e.entity_name, CONCAT("%_", q))
       LIMIT 20
       RETURN e
   )
   
   LET rootCandidate = FIRST(UNION(exact, prefixed))
   
   FOR root IN rootCandidate == null ? [] : [rootCandidate]
     LET contributors = (
       FOR v, edge, path IN 0..6 OUTBOUND root sysml_Relations
         OPTIONS { order: "bfs", uniqueVertices: "global" }
         FILTER LENGTH(path.edges) == LENGTH(
           FOR pe IN path.edges
             FILTER pe.type == "RELATED_TO"
             FILTER pe.relationship_type IN ["owns", "typedby"]
             RETURN 1
         )
         FILTER v.attributes != null
 

# Part 2 — GraphRAG

## 6. `local` — a question in words the model never uses

In [26]:
(await nl.retriever().ask_async(
    "What keeps the astronauts alive and breathing, and what limits does it "
    "have to hold?"
)).show()

Processing local query: What keeps the astronauts alive and breathing, and...
CACHE DISABLED (use_cache=False) - Skipping cache check
LOCAL: local_query start
LOCAL: _retrieve_results_and_context start use_rrf=True rrf_k=20 search_limit=20 final_limit=10
LOCAL: generating embedding for query
LOCAL: embedding generated
LOCAL: executing AQL for primary retrieval
LOCAL: primary retrieval returned 10 results
LOCAL: building context data
LOCAL: context AQL bind_vars: relations_collection='sysml_Relations', nodes_count=10, topChunks=3, topCommunities=3
LOCAL: context nodes=10
LOCAL: formatted context length=76710
LOCAL: retrieval complete results=10 ctx_nodes=10 fmt_len=76710
Created citation_mapping with 7 citations. URLs present: 7
LOCAL: valid citation tokens: [CITE:1], [CITE:2], [CITE:3], [CITE:4], [CITE:5], [CITE:6], [CITE:7]
LOCAL: Final prompt to LLM (59059 chars): # Task
Answer the question using ONLY the Context below. Do not use outside knowledge or add facts the Context does not s

Q  What keeps the astronauts alive and breathing, and what limits does it have to hold?

retrieved  22 documents, 66 edges, 56,609 chars of context

cited (7, first 6)
   {"cite": 1, "source": "models/apollo-11-sysml-v2/Requirements/MissionRequirementsPackage.sysml"}
   {"cite": 2, "source": "models/apollo-11-sysml-v2/Requirements/TechnicalRequirementsPackage.sysml"}
   {"cite": 3, "source": "models/apollo-11-sysml-v2/Technical/SystemSpecificationPackage.sysml"}
   {"cite": 4, "source": "models/apollo-11-sysml-v2/Requirements/MissionRequirementsPackage.sysml"}
   {"cite": 5, "source": "models/apollo-11-sysml-v2/CoSMA/CoSMAQuantitiesAndUnitsPackage.sysml"}
   {"cite": 6, "source": "models/apollo-11-sysml-v2/Requirements/TechnicalRequirementsPackage.sysml"}

A  ## Astronauts' Life Support System

### Portable Life Support System (PLSS)

The Portable Life Support System (PLSS) is responsible for supplying breathable oxygen to the astronauts. The requirement for the PLSS is defined under `

## 7. `unified` — a figure that never became an entity

In [27]:
answer = await nl.retriever().ask_async(
    "How much drinking water must the environmental control system supply "
    "per crew member per day?",
    scope="unified")
answer.show(row_limit=3)
answer.evidence(chars=420, find="water")

Processing unified query: How much drinking water must the environmental con...
CACHE DISABLED (use_cache=False) - Skipping cache check
Unified query parameters: {'final_limit': 10, 'search_limit': 20, 'rrf_k': 20, 'enable_rrf_fusion': True, 'entity_search_limit': 15, 'chunks_limit': 15, 'local_limit': 15, 'enable_chunk_context': True, 'llm_timeout': 60.0, 'response_instructions': 'You are answering questions about one specific set of SysML v2\nmodels, from the retrieved context and nothing else. The context is the model. Your\nown knowledge of Apollo, spacecraft or drones is not evidence and must not appear in\nthe answer, even when it agrees with the context and even when it would fill an\nobvious gap.\n\n- Answer about *these* models, not about the subject in general. Name the elements\n  the context names, using the names the context gives them, and say which source\n  file each came from where the context shows one. An answer that would read the\n  same against any other drone or 

Q  How much drinking water must the environmental control system supply per crew member per day?

retrieved  11 documents, 80 edges, 33,155 chars of context

cited (8, first 3)
   {"cite": 1, "source": "models/apollo-11-sysml-v2/Requirements/TechnicalRequirementsPackage.sysml"}
   {"cite": 2, "source": "models/apollo-11-sysml-v2/Requirements/FunctionalRequirementsPackage.sysml"}
   {"cite": 3, "source": "models/apollo-11-sysml-v2/Technical/SystemSpecificationPackage.sysml"}

A  ### Answer

The environmental control system must supply no less than 2 kilograms of potable water per crew member per day[CITE:1].

evidence  (420 chars at char 2,689, of 33,155 retrieved)
   		}
   	}
   	requirement def <'CLR-R064'> ECSWaterSupplyRate {
   		doc /* The Apollo 11 Mission's crew's potable water system shall provide a minimum of 2 kilograms of potable water per crew member per day. */
   		@Rationale {
   			text = "Adequate potable water is a fundamental requirement for sustaining crew health a

In [28]:
answer = await nl.retriever().ask_async(
    "What is the minimum delta-v the lunar module ascent stage has to provide, "
    "and why that figure?",
    scope="unified")
answer.show(row_limit=3)
answer.evidence(chars=320, find="delta")

Processing unified query: What is the minimum delta-v the lunar module ascen...
CACHE DISABLED (use_cache=False) - Skipping cache check
Unified query parameters: {'final_limit': 10, 'search_limit': 20, 'rrf_k': 20, 'enable_rrf_fusion': True, 'entity_search_limit': 15, 'chunks_limit': 15, 'local_limit': 15, 'enable_chunk_context': True, 'llm_timeout': 60.0, 'response_instructions': 'You are answering questions about one specific set of SysML v2\nmodels, from the retrieved context and nothing else. The context is the model. Your\nown knowledge of Apollo, spacecraft or drones is not evidence and must not appear in\nthe answer, even when it agrees with the context and even when it would fill an\nobvious gap.\n\n- Answer about *these* models, not about the subject in general. Name the elements\n  the context names, using the names the context gives them, and say which source\n  file each came from where the context shows one. An answer that would read the\n  same against any other drone or 

Q  What is the minimum delta-v the lunar module ascent stage has to provide, and why that figure?

retrieved  7 documents, 38 edges, 16,267 chars of context

cited (6, first 3)
   {"cite": 1, "source": "models/apollo-11-sysml-v2/Requirements/TechnicalRequirementsPackage.sysml"}
   {"cite": 2, "source": "models/apollo-11-sysml-v2/Technical/SystemSpecificationPackage.sysml"}
   {"cite": 3, "source": "models/apollo-11-sysml-v2/Requirements/TechnicalRequirementsPackage.sysml"}

A  ### Minimum Delta-V Requirement

The lunar module ascent stage must provide a minimum delta-V of 1,850 m/s. This requirement is specified to achieve lunar orbit, effectively specifying the required propulsive performance needed to escape the Moon's gravitational pull[CITE:1][CITE:3]. 

This figure of 1,850 m/s is defined under the requirement CLR-R115, also documented as LMASCENTDELTAVMINIMUM[CITE:1].

evidence  (320 chars from the start, of 16,267 retrieved)
   115'> LmAscentDeltaVMinimum {
   		doc /* The LM as

## 8. `global` — a question no single row answers

In [29]:
(await nl.retriever().ask_async(
    "What concerns are these models organised around, and what does each part "
    "of the corpus contribute?",
    scope="global")).show()

Processing global query: What concerns are these models organised around, a...
CACHE DISABLED (use_cache=False) - Skipping cache check
Communities collection has embeddings: True
Vector index for embedding already exists
Vector index 'vector_cosine_communities' ensured in 'sysml_Communities' collection.
Using vector search with cosine similarity for community retrieval
Generated query embedding (dimension: 1536)
Vector search returned 88 communities with similarity scores: min=0.1942, max=0.3633, avg=0.2884
Retrieved 88 communities using vector search
Similarity threshold filter (0.25): 88 → 79 communities
Retrieved 79 communities after filtering
Global query completed using vector search. Query: 'What concerns are these models organised around, a...', Final communities: 79
Grouping to 3 groups for global search
Using OpenAI-compatible LLM API for community mapping
Using OpenAI-compatible LLM API for community mapping
Using OpenAI-compatible LLM API for community mapping
JSON data succ

Q  What concerns are these models organised around, and what does each part of the corpus contribute?

retrieved  79 community reports -> 42 points

A  # Overview of Model Concerns and Contributions

The SysML v2 models provided in the dataset are organized around a multitude of concerns, primarily revolving around technical requirements, stakeholder interests, and mission-specific operations. These elements are designed to enhance understanding and alignment across mission scenarios, addressing critical activities, specifications, and roles within mission systems. Below is a detailed synthesis of key areas:

## Core Mission Elements

### Spacecraft Operations
- **SPACECRAFT**: Focused on lunar travel and operations, this component manages parts like the Command/Service Module (CSM) and Lunar Module (LM). It includes mission-critical operations such as `EXECUTEASCENTBURN`, essential for successful mission execution.

- **APOLLO11MISSION**: Provides the core framework by detailing requi